# IFBA Portal Accessibility Analysis

This notebook consumes the result files in `results/` directly (produced by the
collection suite with axe-core and IBM Equal Access Checker) and reproduces the
full accessibility analysis: overall panorama, violations per tool, manual-review
items, distribution by WCAG criterion and POUR principle, impact profile, and
the comparison between the two engines. A closing section validates those
findings against seven live portal pages, testing whether the barriers found on
the homepage are template-driven and recur across the site.

All aggregation logic lives in the tested `a11y` package. Here we only load the
data, assemble the tables, and display them. Re-running this notebook after a new
collection updates every number automatically.

To emit the markdown versions of the tables set `EXPORT = True` in the
configuration cell and re-run.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', None)

cwd = Path.cwd()
if (cwd / 'analysis').exists():
    sys.path.insert(0, str(cwd / 'analysis'))
    base = cwd
else:
    base = cwd.parent

from a11y import aggregate, convergence
from a11y.load import load_results

RESULTS_DIR = base / 'results' / 'archive'
results = load_results(RESULTS_DIR)

DEVICE_LABELS = {'desktop': 'Desktop (1280×720)', 'mobile': 'Mobile (375×667)'}
BUCKET_LABELS = {
    'violation': 'Confirmed violations',
    'potentialviolation': 'Potential violations',
    'recommendation': 'Recommendations',
    'potentialrecommendation': 'Potential recommendations',
    'manual': 'Manual review',
}

EXPORT = False


def caption(text):
    from IPython.display import Markdown, display
    display(Markdown(text))


def show(df):
    from IPython.display import display
    display(df)
    if EXPORT:
        print(df.to_markdown(index=False))

## Execution context

Metadata embedded in each result file, making the run self-contained and
auditable.

In [ ]:
context = pd.DataFrame([
    {
        'Device': DEVICE_LABELS[r.metadata.device],
        'Tool': r.metadata.tool,
        'Tool version': r.metadata.tool_version,
        'Browser': f'{r.metadata.browser} {r.metadata.browser_version}',
        'URL': r.metadata.url,
        'Timestamp': r.metadata.timestamp,
    }
    for r in results
])
show(context)

rule_sets = sorted({s for s in (aggregate.rule_set(r) for r in results) if s})
caption(
    f"**IBM rule set:** {' / '.join(rule_sets)}. The `Tool version` above names the npm "
    'package, which loads its rule engine from a CDN at run time, so the rule set that '
    'produced these IBM findings is a separate fact, recorded in '
    '`raw.summary.ruleArchive`. The collection pins it to a dated deployment rather '
    'than to whichever one IBM published most recently, so a re-run applies the same '
    'rules.'
)

## Overall violation panorama

For axe-core we report distinct rules violated and element-level occurrences.
For IBM, the confirmed violations (items at the *violation* level). The
taxonomies differ, so the numbers should not be compared directly.

In [ ]:
overview = pd.DataFrame(aggregate.overview(results)).rename(columns={
    'device': 'Viewport',
    'axe_rules': 'axe-core (rules)',
    'axe_occurrences': 'axe-core (occurrences)',
    'ibm_violations': 'IBM (violations)',
})
overview['Viewport'] = overview['Viewport'].map(DEVICE_LABELS)
show(overview)

## Violations found by axe-core

Rule, associated WCAG criterion, conformance level, impact assigned by the tool,
and occurrences per *viewport*.

In [ ]:
axe = pd.DataFrame(aggregate.axe_rules(results))
axe['wcag'] = axe['wcag'].str.join(' / ')
axe = axe.rename(columns={
    'rule_id': 'Rule', 'wcag': 'WCAG criterion', 'level': 'Level',
    'impact': 'Impact', 'desktop': 'Desktop', 'mobile': 'Mobile',
})
show(axe)

## Violations found by the IBM Equal Access Checker

IBM organizes its findings under its own rule identifiers, and the WCAG
criterion is assigned from the mapping maintained in the analysis package.

One scoping caveat belongs with this table. The checker ran under its default
`IBM_Accessibility` policy, recorded in `raw.summary.policies` of every result
file, which covers WCAG 2.0, 2.1 and 2.2 levels A and AA plus IBM's own
supplemental requirements. The axe-core run, by contrast, was filtered to the
WCAG 2.2 A and AA tags, so the two engines are not scoped alike and one rule
below sits in the difference. `aria_content_in_landmark` is a violation only
under IBM's supplemental policy: the engine's own `WCAG_2_2` policy classifies
the same failure as a recommendation. Placing every element inside a landmark is
a sufficient technique for criterion 1.3.1 rather than a requirement of it, which
is also why axe-core's equivalent rule, `region`, carries the `best-practice` tag
and never enters this study's tag selection. Read that row as an IBM
best-practice finding, not as a WCAG 2.2 AA violation.

In [ ]:
ibm = pd.DataFrame(aggregate.ibm_rules(results))
ibm['wcag'] = ibm['wcag'].str.join(' / ')
ibm = ibm.rename(columns={
    'rule_id': 'ruleId', 'wcag': 'WCAG criterion', 'level': 'Level',
    'desktop': 'Desktop', 'mobile': 'Mobile',
})
show(ibm)

## Items by category (IBM): manual-review context

Beyond the confirmed violations, IBM reports a substantial volume of items that
require human judgement, taken from the `summary.counts` object.

In [ ]:
buckets = pd.DataFrame(aggregate.ibm_buckets(results))
buckets['category'] = buckets['category'].map(BUCKET_LABELS)
buckets = buckets.rename(columns={'category': 'Category', 'desktop': 'Desktop', 'mobile': 'Mobile'})
show(buckets)

## Distribution by WCAG criterion

Occurrences aggregated by WCAG criterion and the corresponding POUR principle,
for each engine. A rule that its taxonomy binds to more than one criterion
contributes its full count to each of them, so the totals here and in the two
subsections that follow exceed the occurrence counts of the panorama table above.
The unit of these tables is the criterion, not the occurrence.

In [ ]:
def criterion_table(tool):
    df = pd.DataFrame(aggregate.by_criterion(results, tool))
    return df.rename(columns={
        'criterion': 'WCAG criterion', 'principle': 'Principle',
        'level': 'Level', 'occurrences': 'Occurrences',
    })

caption('**axe-core**')
show(criterion_table('axe-core'))
caption('**IBM Equal Access Checker**')
show(criterion_table('ibm-equal-access'))

### Aggregation by POUR principle

The criterion totals above rolled up into the four WCAG principles (Perceivable,
Operable, Understandable, Robust). They carry the multiple-criterion counting
noted above, so they sum to the same inflated total.

In [ ]:
def principle_table(tool):
    df = pd.DataFrame(aggregate.by_principle(results, tool))
    return df.rename(columns={'principle': 'Principle', 'occurrences': 'Occurrences'})

caption('**axe-core**')
show(principle_table('axe-core'))
caption('**IBM Equal Access Checker**')
show(principle_table('ibm-equal-access'))

### Aggregation by conformance level

The same criterion totals rolled up into conformance levels A and AA, the two
levels evaluated in this study.

In [ ]:
def level_table(tool):
    df = pd.DataFrame(aggregate.by_level(results, tool))
    return df.rename(columns={'level': 'Level', 'occurrences': 'Occurrences'})

caption('**axe-core**')
show(level_table('axe-core'))
caption('**IBM Equal Access Checker**')
show(level_table('ibm-equal-access'))

## Impact profile (axe-core)

Distribution of axe-core occurrences by the impact level assigned by the tool.
IBM does not classify findings by severity, so the impact profile is based on
axe-core.

In [ ]:
impact = pd.DataFrame(aggregate.by_impact(results)).rename(
    columns={'impact': 'Impact', 'occurrences': 'Occurrences'})
show(impact)

## Convergences and divergences between the engines

Since axe-core and IBM use distinct rule taxonomies, the comparison is made by
WCAG criterion: criteria flagged by both engines (convergence) and criteria
exclusive to each engine (tool-specific blind spots).

A second table pairs the two taxonomies at the rule level, so each barrier
shows the identifier each engine gives it and whether both engines see it.


In [ ]:
partition = convergence.compare(results)
comparison = pd.DataFrame([
    {'Group': 'Both engines', 'WCAG criteria': ' / '.join(partition['shared'])},
    {'Group': 'axe-core only', 'WCAG criteria': ' / '.join(partition['axe_only'])},
    {'Group': 'IBM only', 'WCAG criteria': ' / '.join(partition['ibm_only'])},
])
show(comparison)

In [ ]:
RULE_PAIR_LABELS = {
    'both': 'Both engines',
    'axe_only': 'axe-core only',
    'ibm_only': 'IBM only',
}

pairs = pd.DataFrame(convergence.rule_pairs(results))
pairs['criteria'] = pairs['criteria'].str.join(' / ')
pairs['detection'] = pairs['detection'].map(RULE_PAIR_LABELS)
pairs = pairs.fillna('---').rename(columns={
    'criteria': 'WCAG criterion',
    'level': 'Level',
    'axe_rule': 'axe-core rule',
    'ibm_rule': 'IBM ruleId',
    'detection': 'Detection',
})
show(pairs[['WCAG criterion', 'Level', 'axe-core rule', 'IBM ruleId', 'Detection']])


## Desktop vs. mobile comparison

WCAG criteria flagged in each browsing context, considering both engines
together: criteria common to both *viewports* and criteria specific to each
modality. Criterion-level presence is coarser than the occurrence counts: a
component rendered in only one of the two layouts can move a rule's count sharply
while leaving its criterion flagged in both contexts, so the partition below and
the per-rule tables above answer different questions.

In [ ]:
partition = aggregate.viewport_criteria(results)
viewports = pd.DataFrame([
    {'Group': 'Both viewports', 'WCAG criteria': ' / '.join(partition['shared'])},
    {'Group': 'Desktop only', 'WCAG criteria': ' / '.join(partition['desktop_only'])},
    {'Group': 'Mobile only', 'WCAG criteria': ' / '.join(partition['mobile_only'])},
])
show(viewports)

## User impact of the barriers

Each WCAG criterion flagged on the archived homepage, mapped to the user groups
that the "Benefits" rationale of the WCAG 2.2 *Understanding* documents names as
directly affected. This reframes the occurrence counts as who is impacted and why
each barrier matters, and it feeds the prioritization that follows.

In [ ]:
GROUP_LABELS = {
    'screen_reader': 'Blind / screen-reader users',
    'low_vision': 'Low vision',
    'color_vision_deficiency': 'Color-vision deficiency',
    'older_users': 'Older users',
    'motor': 'Motor / dexterity (e.g. tremor)',
    'touch_mobile': 'Touch / mobile users',
    'cognitive': 'Cognitive, language and learning',
}

user_impact = pd.DataFrame(aggregate.impact_table(results))
user_impact['groups'] = user_impact['groups'].apply(
    lambda g: ', '.join(GROUP_LABELS[t] for t in g)
)
user_impact = user_impact.rename(columns={
    'criterion': 'WCAG criterion', 'principle': 'Principle',
    'level': 'Level', 'groups': 'Affected user groups',
})
show(user_impact)

## Prioritization for recommendations

axe-core rules ordered by impact severity, then by WCAG conformance level (A
before AA), then by occurrence frequency. This ordering supports proposing
improvement recommendations for the highest-impact, most fundamental, and most
frequent violations first. IBM does not classify findings by severity, so the
prioritization is based on axe-core.

In [ ]:
prio = pd.DataFrame(aggregate.priorities(results))
prio['wcag'] = prio['wcag'].str.join(' / ')
prio = prio.rename(columns={
    'rule_id': 'Rule', 'wcag': 'WCAG criterion', 'level': 'Level',
    'impact': 'Impact', 'occurrences': 'Occurrences',
})
show(prio)

## Summary of findings

Consolidation of the main figures from the previous sections. The values are
computed from the loaded results, so they track new collections automatically.

In [ ]:
ov = {r['device']: r for r in aggregate.overview(results)}
conv = convergence.compare(results)
vp = aggregate.viewport_criteria(results)
principles = [p['principle'] for p in aggregate.by_principle(results, 'axe-core') if p['occurrences']]
top = aggregate.priorities(results)[0]

summary = f'''
- **Panorama:** axe-core flagged {ov['desktop']['axe_rules']} rule(s) and {ov['desktop']['axe_occurrences']} occurrence(s) on desktop and {ov['mobile']['axe_rules']} rule(s) and {ov['mobile']['axe_occurrences']} occurrence(s) on mobile. IBM confirmed {ov['desktop']['ibm_violations']} and {ov['mobile']['ibm_violations']} violations, respectively.
- **POUR principles affected (axe-core):** {', '.join(principles)}.
- **Engine complementarity:** {len(conv['shared'])} criterion(s) flagged by both, {len(conv['axe_only'])} exclusive to axe-core and {len(conv['ibm_only'])} exclusive to IBM.
- **Browsing context:** {len(vp['desktop_only'])} criterion(s) exclusive to desktop and {len(vp['mobile_only'])} to mobile.
- **Correction priority:** the highest-impact, most-frequent rule is `{top['rule_id']}` ({top['impact']}, {top['occurrences']} occurrences).
'''
caption(summary)

## Live validation across selected portal pages

The analysis above targets the archived portal homepage, the reproducible
backbone that stays comparable to the prior IFBA-portal studies, all of which
evaluated homepages only. As an external-validity check we additionally evaluate
seven **live** portal pages, none of which redirect during the *Defeso Eleitoral*
embargo, to test whether the homepage's barriers are template-driven and recur
across the live site.

The pages vary along the dimensions that matter for that claim: the **Salvador**
campus homepage (flagship, template-similar); the **Euclides da Cunha** and
**Feira de Santana** campus homepages (independently maintained, and instantiating
the same Galleria media-carousel the portal homepage carries); an **institutional
news article** (an instance of the portal's most-instantiated news template); the
institutional **Access-to-Information** page (a data-table component class no other
measured page contains); and the two **Ingresso 2026** pages, the selection landing
page and its publications listing, which form the highest-traffic page class after
the homepage. Occurrence counts are not comparable across different pages, so the
unit of the claim is the set of violated rules and the components their selectors
point to. The table below records the URL and capture timestamp of each page, so
the comparability of the set can be checked against the data rather than assumed.

In [ ]:
LIVE_PAGE_LABELS = {
    'salvador': 'Salvador campus homepage',
    'euclides-da-cunha': 'Euclides da Cunha campus homepage',
    'feira-de-santana': 'Feira de Santana campus homepage',
    'noticia-institucional': 'Institutional news article',
    'acessoainformacao-institucional': 'Access-to-Information page',
    'ingresso-2026': 'Ingresso 2026 selection page',
    'ingresso-2026-publicacoes': 'Ingresso 2026 publications page',
}
LIVE_DIR = base / 'results' / 'live'
live_by_page = {label: load_results(LIVE_DIR / label) for label in LIVE_PAGE_LABELS}
live_results = live_by_page['salvador']

live_context = pd.DataFrame([
    {
        'Page': LIVE_PAGE_LABELS[label],
        'URL': res[0].metadata.url,
        'Timestamp': res[0].metadata.timestamp,
    }
    for label, res in live_by_page.items()
])
show(live_context)

### Violations on the Salvador campus homepage (primary check)

Salvador is presented in full as the representative external-validity page, in
the same format as the portal-homepage tables above. The other live pages are
summarized in the propagation matrix further below.

In [ ]:
live_overview = pd.DataFrame(aggregate.overview(live_results)).rename(columns={
    'device': 'Viewport',
    'axe_rules': 'axe-core (rules)',
    'axe_occurrences': 'axe-core (occurrences)',
    'ibm_violations': 'IBM (violations)',
})
live_overview['Viewport'] = live_overview['Viewport'].map(DEVICE_LABELS)
show(live_overview)

live_axe = pd.DataFrame(aggregate.axe_rules(live_results))
live_axe['wcag'] = live_axe['wcag'].str.join(' / ')
live_axe = live_axe.rename(columns={
    'rule_id': 'Rule', 'wcag': 'WCAG criterion', 'level': 'Level',
    'impact': 'Impact', 'desktop': 'Desktop', 'mobile': 'Mobile',
})
caption('**axe-core**')
show(live_axe)

live_ibm = pd.DataFrame(aggregate.ibm_rules(live_results))
live_ibm['wcag'] = live_ibm['wcag'].str.join(' / ')
live_ibm = live_ibm.rename(columns={
    'rule_id': 'ruleId', 'wcag': 'WCAG criterion', 'level': 'Level',
    'desktop': 'Desktop', 'mobile': 'Mobile',
})
caption('**IBM Equal Access Checker**')
show(live_ibm)

### Rule-set overlap with the portal homepage

Partition of the violated rules between the two pages: rules flagged on **both**
(the shared template family), rules exclusive to the portal homepage, and rules
exclusive to the Salvador campus homepage. Because the pages differ, this set
relationship, and not the raw occurrence counts, is what the validation rests on.

In [ ]:
from a11y import validation


def overlap_table(tool):
    p = validation.target_rules(results, live_results, tool)
    return pd.DataFrame([
        {'Group': 'Both pages (shared template family)', 'Rules': ' / '.join(p['shared'])},
        {'Group': 'Portal homepage only', 'Rules': ' / '.join(p['reference_only'])},
        {'Group': 'Salvador campus homepage only', 'Rules': ' / '.join(p['live_only'])},
    ])

caption('**axe-core**')
show(overlap_table('axe-core'))
caption('**IBM Equal Access Checker**')
show(overlap_table('ibm-equal-access'))

axe_p = validation.target_rules(results, live_results, 'axe-core')
ibm_p = validation.target_rules(results, live_results, 'ibm-equal-access')
axe_home = len(axe_p['shared']) + len(axe_p['reference_only'])
ibm_home = len(ibm_p['shared']) + len(ibm_p['reference_only'])
new_rules = len(axe_p['live_only']) + len(ibm_p['live_only'])
absent = axe_p['reference_only'] + ibm_p['reference_only']
absent_note = (
    f" The rules flagged only on the portal homepage (`{'`, `'.join(absent)}`) are not"
    ' reproduced there.'
    if absent else
    ' Every rule flagged on the portal homepage is reproduced there as well.'
)
caption(f'''
The live Salvador campus homepage reproduces **{len(axe_p['shared'])} of {axe_home}** axe-core rules and **{len(ibm_p['shared'])} of {ibm_home}** IBM rules also flagged on the portal homepage, and introduces **{new_rules}** rule(s) not seen there.{absent_note} The overlap confirms the toolchain runs against the live production server and that a template-similar live page surfaces the same family of barriers.
''')

### Component-level attribution from selectors

The rule-set overlap establishes *which* rules recur across the two pages. The
`targets` selectors stored with every violation let us go one level finer and ask
*which elements* each rule attaches to, separating barriers that recur across pages
from those confined to page-specific content.

For each rule we partition the violated selectors into those flagged on **both**
pages and those exclusive to one of them, per *viewport*. Two cautions apply to
reading this partition. A selector flagged identically on two different pages shows
that the barrier recurs across pages rather than being confined to one, which is
consistent with an origin in the shared Plone templates or in a widget reused on
both pages, but the selector alone does not establish which of the two. Conversely,
selectors are DOM paths, so an element genuinely reused across pages can still be
reported under different selector strings when its surrounding markup differs. The
shared count is therefore a lower bound on cross-page recurrence, not an exhaustive
measure of it.

That lower bound is degenerate for the IBM tables. IBM reports each occurrence as
an absolute XPath, which encodes the whole ancestor chain, so two different pages
cannot produce the same string and every IBM rule shows zero shared selectors
whatever recurs between them. Only axe's CSS selectors, anchored on ids and
classes, can coincide across pages. Cross-page recurrence under IBM is established
by the rule-set overlap above and by the component attribution further below,
not by this partition.

In [ ]:
def selector_table(tool, device):
    partition = validation.target_selectors(results, live_results, tool, device)
    return pd.DataFrame([
        {
            'Rule': rule_id,
            'Both pages': len(groups['shared']),
            'Portal only': len(groups['reference_only']),
            'Salvador only': len(groups['live_only']),
            'Selectors on both pages': ' / '.join(groups['shared']),
        }
        for rule_id, groups in partition.items()
    ])

for tool_label, tool in (('axe-core', 'axe-core'), ('IBM Equal Access Checker', 'ibm-equal-access')):
    for device in ('desktop', 'mobile'):
        caption(f'**{tool_label}, {DEVICE_LABELS[device]}**')
        show(selector_table(tool, device))

lines = []
for device in ('desktop', 'mobile'):
    partition = validation.target_selectors(results, live_results, 'axe-core', device)
    for rule_id, groups in partition.items():
        if groups['shared']:
            selectors = '`, `'.join(groups['shared'])
            lines.append(
                f"- `{rule_id}` ({device}): {len(groups['shared'])} shared selector(s) --- `{selectors}`"
            )
caption(
    'axe-core selectors flagged on both the portal homepage and the live Salvador '
    'campus homepage. These barriers recur across the two pages rather than being '
    'confined to the homepage:\n\n' + '\n'.join(lines)
)


### Template propagation across the selected live pages

The tables below place all seven live pages beside the archived homepage. The
first pair reports occurrences per rule for each engine on *desktop*, so a rule
filled across columns is a template-level barrier rather than a homepage
artifact. The overlap summary then partitions each page's rules into those shared
with the homepage, those only the homepage carries, and any rule new to that
page.


In [ ]:

PAGES = {'Homepage (arch.)': results}
PAGES.update({LIVE_PAGE_LABELS[label]: res for label, res in live_by_page.items()})


def occurrence_matrix(tool, device):
    data = {}
    for page, res in PAGES.items():
        counts = {}
        for r in res:
            if r.metadata.tool != tool or r.metadata.device != device:
                continue
            for v in r.violations:
                counts[v.rule_id] = counts.get(v.rule_id, 0) + v.count
        data[page] = counts
    df = pd.DataFrame(data).fillna(0).astype(int)
    order = (df > 0).sum(axis=1).sort_values(ascending=False, kind='stable').index
    df = df.loc[order]
    df.index.name = 'Rule'
    return df.reset_index()


for tool_label, tool in (('axe-core', 'axe-core'), ('IBM Equal Access Checker', 'ibm-equal-access')):
    caption(f'**{tool_label} --- occurrences per rule and page (desktop)**')
    show(occurrence_matrix(tool, 'desktop'))


In [ ]:

def overlap_summary():
    rows = []
    for label, res in live_by_page.items():
        for tool_label, tool in (('axe', 'axe-core'), ('IBM', 'ibm-equal-access')):
            p = validation.target_rules(results, res, tool)
            rows.append({
                'Page': LIVE_PAGE_LABELS[label],
                'Engine': tool_label,
                'Shared': len(p['shared']),
                'Homepage-only': len(p['reference_only']),
                'New on this page': ' / '.join(p['live_only']) or '---',
            })
    return pd.DataFrame(rows)


show(overlap_summary())


### Reading the propagation matrix

The matrix separates three levels at which a barrier can originate, which is the
distinction the propagation claim rests on.

A rule filled across every column originates at the **template level**, in markup
the Plone theme emits on every page, and is therefore independent of what any
single page publishes. A rule filled only where a page instantiates a given
component originates at the **component level**, in that component's own markup or
in the plugin behind it; its count then tracks how much of the component each page
configures, so a page can carry the component and still show nothing when the
content it holds happens to be compliant. A rule appearing on pages with no
component in common originates at the **editorial level**, in what authors write
and embed page by page, and it follows the content rather than the platform.

Two cautions apply. Absent occurrences do not establish that a page omits a
component, only that the page surfaced no failure of it, so a zero is evidence
about the measurement and not about the markup. And the overlap summary counts
rules rather than occurrences, so a page sharing every rule with the homepage may
still differ widely in how many elements each rule reaches.

Assigning a rule to one of the three levels is what the selectors decide, not the
counts. The cell below prints the selectors the engines flagged for the rules that
distinguish the pages, and the component table further down decomposes the most
frequent rule by the component each selector belongs to.

In [ ]:
def selectors_for(label, tool, rule, device='desktop', n=3):
    for r in live_by_page.get(label, []):
        if r.metadata.tool == tool and r.metadata.device == device:
            for v in r.violations:
                if v.rule_id == rule:
                    return v.targets[:n]
    return []


evidence = {
    'Euclides carousel --- image-alt (axe)':
        selectors_for('euclides-da-cunha', 'axe-core', 'image-alt'),
    'Access-to-Info table --- table_headers_exists (IBM)':
        selectors_for('acessoainformacao-institucional', 'ibm-equal-access', 'table_headers_exists', n=1),
    'Access-to-Info links --- link-in-text-block (axe)':
        selectors_for('acessoainformacao-institucional', 'axe-core', 'link-in-text-block'),
    'News iframe --- frame-title (axe)':
        selectors_for('noticia-institucional', 'axe-core', 'frame-title', n=1),
}
found = {k: v for k, v in evidence.items() if v}
caption(
    '\n'.join(f"- **{k}**: `{'`, `'.join(v)}`" for k, v in found.items())
    if found else
    'None of the rules probed for selector evidence were flagged on the pages probed.'
)

### Component attribution of the target-size occurrences

The propagation matrix reports `target-size` as a single number per page. The
table below decomposes it by the component each flagged selector belongs to,
which is what separates a barrier carried by the shared template from one carried
by a component a page chose to instantiate. A component holding the same count on
every page sits at the template level, and one whose count varies tracks how much
of itself each page configures. A component absent from one *viewport* only means
the engine flagged nothing for it there, which may be because the layout does not
render it or because it renders and passes; separating those two calls for
inspecting the rendered page, not this table.

In [ ]:
from a11y import components


COMPONENT_LABELS = {
    'banner_rotator': 'Banner rotator controls',
    'grid_footer': 'Grid footer (doormat)',
    'news_tile': 'News-tile link',
    'other': 'Other',
}


def component_matrix(rule, device):
    data = {}
    for page, res in PAGES.items():
        rows = components.component_counts(res, rule, device)
        data[page] = {COMPONENT_LABELS[row['component']]: row['occurrences'] for row in rows}
    df = pd.DataFrame(data).fillna(0).astype(int)
    df.index.name = 'Component'
    return df.reset_index()


for device in ('desktop', 'mobile'):
    caption(f'**`target-size` occurrences by component ({device})**')
    show(component_matrix('target-size', device))
